# Nexus PnL attribution

**Gross** comes from `algo-research/dashboard/pnl_attribution` (`PnlAttribution`) — same library that feeds `algo.nexus_pnl_attribution`.

Then we decompose that gross into:

$$
\mathrm{Gross} = \mathrm{Overnight} + \mathrm{MarketRisk} + \mathrm{InternalMarkBasis} + \mathrm{PureExecution} + \mathrm{Unexplained}
$$

$$
\mathrm{MarketRisk} = \mathrm{BODInventory} + \mathrm{TransferMark} + \mathrm{HedgesMark}
$$

$$
\mathrm{PureExecution} = \mathrm{MidToTouch} + \mathrm{TouchToExec}
$$

- `$` and `c/bbl` for every component
- `c/bbl = 100 \times \$ / (V \times 1000)` with shared day volume $V = |\mathrm{transfers}| + |\mathrm{hedges}|$ (event-level lots)
- `Unexplained` bridges library gross to the lifecycle sum (inventory not explained by today's transfers + hedges)


In [ ]:
import os
import sys
from datetime import date, datetime, time, timedelta
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

DASHBOARD_ROOT = Path('/home/ajain/apeksha/algo-research/dashboard')
_dash_site = next(DASHBOARD_ROOT.joinpath('.venv/lib').glob('python*/site-packages'))
# Path setup BEFORE importing clickhouse_connect / dashboard packages
sys.path.insert(0, str(_dash_site))
sys.path.insert(0, str(DASHBOARD_ROOT))

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from clickhouse_connect import get_client  # noqa: E402

from dashboard_constants import BOOK_NAMES, CONTRACT_SIZE  # noqa: E402
from pnl_attribution.attribution import PnlAttribution  # noqa: E402
from pnl_attribution.utils import (  # noqa: E402
    months_between,
    parse_end_month,
    parse_front_month,
    sign_trade_quantities,
    sign_transfer_quantities,
)
from data_loaders.positions.positions import (  # noqa: E402
    NexusTradeService,
    NexusTransferService,
    PositionService,
)
from data_loaders.prices.brent_algo_curve import BrentAlgoCurveService  # noqa: E402

client = get_client(
    host=os.getenv('CLICKHOUSE_HOST', '10.20.0.243'),
    port=int(os.getenv('CLICKHOUSE_PORT', '8123')),
    username=os.getenv('CLICKHOUSE_USERNAME', ''),
    password=os.getenv('CLICKHOUSE_PASSWORD', ''),
    database=os.getenv('CLICKHOUSE_DATABASE', 'algo'),
)

print('CONTRACT_SIZE =', CONTRACT_SIZE, '| books =', len(BOOK_NAMES))


In [ ]:
DATE = (datetime.now() - timedelta(days=1)).date()
while DATE.weekday() >= 5:
    DATE -= timedelta(days=1)

# DATE = date(2026, 8, 4)  # uncomment to pin a session date
START = datetime.combine(DATE, time.min)
END = datetime.combine(DATE, time(23, 59, 59))
FREQ = '1h'  # coarse grid; we only need EOD last row

print('DATE =', DATE)


## 1. Library gross (source of truth)

`gross = overnight + m2m_pnl + trade_pnl` from `PnlAttribution.calculate()`.


In [ ]:
attr = PnlAttribution(
    book_names=BOOK_NAMES,
    start_datetime=START,
    end_datetime=END,
    client=client,
)
library_results = attr.calculate(freq=FREQ)

rows = []
for book, data in library_results.items():
    pnl = data['pnl']
    if pnl.empty:
        continue
    eod = pnl.iloc[-1]
    rows.append({
        'book': book,
        'overnight': float(eod['overnight']),
        'm2m_pnl': float(eod['m2m_pnl']),
        'trade_pnl': float(eod['trade_pnl']),
        'library_gross': float(eod['gross']),
    })

library = pd.DataFrame(rows).set_index('book').sort_index()
library.loc['TEAM TOTAL'] = library.sum(numeric_only=True)
library.round(2)


## 2. Lifecycle components

Same dashboard data loaders for positions / curve / trades / transfers, plus TTMDS mid/touch for execution attribution.


In [ ]:
pos_svc = PositionService(client)
curve_svc = BrentAlgoCurveService(client)
trade_svc = NexusTradeService(client)
xfer_svc = NexusTransferService(client)

prices_start = START - pd.Timedelta(days=3)
spread_prices = curve_svc.get_spread_range_wide(prices_start, END)
spread_prices = spread_prices[~spread_prices.index.duplicated(keep='last')]
spread_positions = pos_svc.get_spread_positions(BOOK_NAMES, prices_start, END)

print('curves', spread_prices.shape, '| books with positions', list(spread_positions))


In [ ]:
def curve_at(ts, tenors, curves: pd.DataFrame) -> pd.Series:
    i = curves.index.get_indexer([pd.Timestamp(ts)], method='ffill')[0]
    if i < 0:
        return pd.Series(0.0, index=list(tenors))
    row = curves.iloc[i]
    return pd.Series({t: float(row[t]) if t in row.index and pd.notna(row[t]) else 0.0 for t in tenors})


def get_ttmds_spread_quotes(asof_date, pairs: list[tuple]) -> pd.DataFrame:
    if not pairs:
        return pd.DataFrame(columns=['timestamp', 'start_tenor', 'end_tenor', 'bid', 'ask', 'mid'])

    pair_sql = ', '.join(
        f"(toDate('{pd.Timestamp(s).date()}'), toDate('{pd.Timestamp(e).date()}'))"
        for s, e in sorted(set((pd.Timestamp(a).date(), pd.Timestamp(b).date()) for a, b in pairs))
    )
    quotes = client.query_df(f"""
        SELECT
            toStartOfSecond(timestamp) AS timestamp,
            start_tenor, end_tenor,
            argMax(bid_price, timestamp) AS bid,
            argMax(ask_price, timestamp) AS ask,
            argMax((bid_price + ask_price) / 2, timestamp) AS mid
        FROM ttmds.brn_v2
        WHERE toDate(timestamp) = '{asof_date}'
          AND type = 'S'
          AND bid_price > 0 AND ask_price > bid_price
          AND (start_tenor, end_tenor) IN ({pair_sql})
        GROUP BY timestamp, start_tenor, end_tenor
        ORDER BY start_tenor, end_tenor, timestamp
    """)
    if quotes.empty:
        return quotes
    quotes['timestamp'] = pd.to_datetime(quotes['timestamp'], utc=True).dt.tz_convert('Europe/London')
    quotes['start_tenor'] = pd.to_datetime(quotes['start_tenor']).dt.normalize()
    quotes['end_tenor'] = pd.to_datetime(quotes['end_tenor']).dt.normalize()
    return quotes


def asof_strip_quotes(trades: pd.DataFrame, quotes: pd.DataFrame) -> pd.DataFrame:
    empty = pd.DataFrame({'bid': np.nan, 'ask': np.nan, 'mid': np.nan}, index=trades.index)
    if trades.empty or quotes.empty:
        return empty

    left = trades[['timestamp', 'start_tenor', 'end_tenor']].copy()
    left['timestamp'] = pd.to_datetime(left['timestamp'], utc=True).dt.tz_convert('Europe/London').astype('datetime64[ns, Europe/London]')
    left['start_tenor'] = pd.to_datetime(left['start_tenor']).dt.normalize()
    left['end_tenor'] = pd.to_datetime(left['end_tenor']).dt.normalize()
    left = left.reset_index(drop=True)
    left['_i'] = np.arange(len(left))

    right = quotes.copy()
    right['timestamp'] = pd.to_datetime(right['timestamp'], utc=True).dt.tz_convert('Europe/London').astype('datetime64[ns, Europe/London]')

    pieces = []
    for (s, e), grp in left.groupby(['start_tenor', 'end_tenor'], sort=False):
        md = right[(right['start_tenor'] == s) & (right['end_tenor'] == e)]
        if md.empty:
            out = grp[['_i']].copy()
            out['bid'] = out['ask'] = out['mid'] = np.nan
            pieces.append(out)
            continue
        merged = pd.merge_asof(
            grp.sort_values('timestamp'),
            md[['timestamp', 'bid', 'ask', 'mid']].sort_values('timestamp'),
            on='timestamp',
            direction='backward',
        )
        pieces.append(merged[['_i', 'bid', 'ask', 'mid']])

    return pd.concat(pieces, ignore_index=True).set_index('_i').sort_index()[['bid', 'ask', 'mid']]


def lifecycle_components(book_name: str, asof_date: date) -> dict:
    """Dollar lifecycle components. Overnight is sanity-only; tree uses library overnight."""
    if book_name not in spread_positions:
        return {'book': book_name, 'error': 'no positions'}

    p = spread_positions[book_name].groupby(level=0).last().sort_index()
    curves = spread_prices.groupby(level=0).last().sort_index()

    eod_p = p.groupby(p.index.date).tail(1)
    eod_p.index = eod_p.index.date
    prev_eod_pos = eod_p.shift()

    eod_c = curves.groupby(curves.index.date).tail(1)
    eod_c.index = eod_c.index.date
    prev_eod_c = eod_c.shift()

    bod_c = curves.groupby(curves.index.date).head(1)
    bod_c.index = bod_c.index.date

    day_curves = curves[curves.index.date == asof_date]
    day_pos = p[p.index.date == asof_date]
    if day_curves.empty or asof_date not in prev_eod_pos.index or asof_date not in bod_c.index:
        return {'book': book_name, 'error': 'missing BOD/EOD inputs'}

    trades = trade_svc.get_range(book_name, START, END)
    xfers = xfer_svc.get_range(book_name, START, END)
    if not trades.empty:
        trades = trades[pd.to_datetime(trades['timestamp']).dt.date == asof_date].copy()
    if not xfers.empty:
        xfers = xfers[pd.to_datetime(xfers['timestamp']).dt.date == asof_date].copy()

    if not trades.empty:
        trades = sign_trade_quantities(trades)
    if not xfers.empty:
        xfers = sign_transfer_quantities(xfers)

    xfer_events, trade_rows, pairs = [], [], []
    abs_transfer_lots = abs_hedge_lots = 0.0

    if not xfers.empty:
        for _, row in xfers.iterrows():
            q = float(row['quantity'])
            abs_transfer_lots += abs(q)
            tenors = months_between(pd.Timestamp(row['start_tenor']), pd.Timestamp(row['end_tenor']))
            if tenors:
                xfer_events.append({'q': q, 'tenors': tenors, 'price': float(row['price'])})

    if not trades.empty:
        for _, row in trades.iterrows():
            q = float(row['quantity'])
            try:
                start = parse_front_month(row['instrument_key'])
                end = parse_end_month(row['instrument_key'])
                tenors = months_between(start, end)
            except Exception:
                continue
            if not tenors:
                continue
            abs_hedge_lots += abs(q)
            pairs.append((start, end))
            trade_rows.append({
                'timestamp': row['timestamp'], 'start_tenor': start, 'end_tenor': end,
                'tenors': tenors, 'q': q, 'price': float(row['price']),
            })

    trade_df = pd.DataFrame(trade_rows)
    quotes = get_ttmds_spread_quotes(asof_date, pairs)
    if not trade_df.empty:
        qdf = asof_strip_quotes(trade_df, quotes)
        trade_df['ttmds_bid'] = qdf['bid'].values
        trade_df['ttmds_ask'] = qdf['ask'].values
        trade_df['ttmds_mid'] = qdf['mid'].values

    event_tenors = {t for e in xfer_events for t in e['tenors']}
    if not trade_df.empty:
        event_tenors |= {t for tenors in trade_df['tenors'] for t in tenors}

    Xt, X0 = day_curves.iloc[-1], bod_c.loc[asof_date]
    Q0, Qt = prev_eod_pos.loc[asof_date], day_pos.iloc[-1]
    cols = sorted(set(Q0.index) | set(Qt.index) | set(Xt.index) | set(X0.index) | event_tenors)
    Q0, Qt, Xt, X0 = [s.reindex(cols).fillna(0.0) for s in (Q0, Qt, Xt, X0)]

    if asof_date in prev_eod_c.index:
        Xprev = prev_eod_c.loc[asof_date].reindex(cols).fillna(0.0)
        overnight_check = float((CONTRACT_SIZE * Q0 * (X0 - Xprev)).sum())
    else:
        overnight_check = 0.0

    bod_inventory = float((CONTRACT_SIZE * Q0 * (Xt - X0)).sum())

    flow = pd.Series(0.0, index=cols)
    transfer_mark = 0.0
    for e in xfer_events:
        q, tenors, pj = e['q'], e['tenors'], e['price']
        n = len(tenors)
        for t in tenors:
            flow.loc[t] += q
            transfer_mark += CONTRACT_SIZE * q * (float(Xt[t]) - pj / n)

    post_fill = mark_basis = mid_to_touch = touch_to_exec = 0.0
    if not trade_df.empty:
        for _, e in trade_df.iterrows():
            q, tenors, E = e['q'], e['tenors'], e['price']
            Xi = curve_at(e['timestamp'], tenors, curves).reindex(tenors).fillna(0.0)
            Xi_sum = float(Xi.sum())

            P, bid, ask = e['ttmds_mid'], e['ttmds_bid'], e['ttmds_ask']
            if pd.isna(P) or pd.isna(bid) or pd.isna(ask):
                P = T = Xi_sum
            else:
                T = float(ask) if q > 0 else float(bid)
                P = float(P)

            mark_basis += CONTRACT_SIZE * q * (Xi_sum - P)
            mid_to_touch += CONTRACT_SIZE * q * (P - T)
            touch_to_exec += CONTRACT_SIZE * q * (T - E)
            for t in tenors:
                flow.loc[t] += q
                post_fill += CONTRACT_SIZE * q * (float(Xt[t]) - float(Xi[t]))

    market_risk = bod_inventory + transfer_mark + post_fill
    pure_exec = mid_to_touch + touch_to_exec
    pos_gap = Qt - (Q0 + flow)
    pos_gap_pnl = float((CONTRACT_SIZE * pos_gap * Xt).sum())

    return {
        'book': book_name,
        'overnight_check': overnight_check,
        'bod_inventory': bod_inventory,
        'transfer_mark': transfer_mark,
        'hedges_mark': post_fill,
        'market_risk': market_risk,
        'internal_mark_basis': mark_basis,
        'mid_to_touch': mid_to_touch,
        'touch_to_exec': touch_to_exec,
        'pure_execution': pure_exec,
        'pos_gap_pnl': pos_gap_pnl,
        'volume_lots': abs_transfer_lots + abs_hedge_lots,
        'abs_transfer_lots': abs_transfer_lots,
        'abs_hedge_lots': abs_hedge_lots,
        'ttmds_fills_matched': int(trade_df['ttmds_mid'].notna().sum()) if not trade_df.empty else 0,
        'ttmds_fills_total': len(trade_df),
    }


In [ ]:
life_rows = [lifecycle_components(b, DATE) for b in BOOK_NAMES]
life = pd.DataFrame(life_rows).set_index('book')
life


## 3. Combined tree — `$` and `c/bbl`

Uses **library overnight + library gross**. Lifecycle fills the middle. `unexplained = library_gross − explained`.


In [ ]:
COMPONENT_ORDER = [
    'total',
    'overnight',
    'market_risk',
    '  bod_inventory',
    '  transfer_mark',
    '  hedges_mark',
    'internal_mark_basis',
    'pure_execution',
    '  mid_to_touch',
    '  touch_to_exec',
    'unexplained',
]


def dollars_to_c_bbl(dollars: float, volume_lots: float) -> float:
    if volume_lots is None or volume_lots <= 0 or pd.isna(volume_lots) or pd.isna(dollars):
        return np.nan
    return 100.0 * dollars / (volume_lots * CONTRACT_SIZE)


books = sorted((set(library.index) - {'TEAM TOTAL'}) & set(life.index))
combined = []
for book in books:
    lib = library.loc[book]
    lc = life.loc[book]
    if 'error' in life.columns and pd.notna(lc.get('error')):
        continue

    overnight = float(lib['overnight'])
    market_risk = float(lc['market_risk'])
    mark_basis = float(lc['internal_mark_basis'])
    pure_exec = float(lc['pure_execution'])
    explained = overnight + market_risk + mark_basis + pure_exec
    library_gross = float(lib['library_gross'])
    unexplained = library_gross - explained
    vol = float(lc['volume_lots'])

    dollar = {
        'total': library_gross,
        'overnight': overnight,
        'market_risk': market_risk,
        '  bod_inventory': float(lc['bod_inventory']),
        '  transfer_mark': float(lc['transfer_mark']),
        '  hedges_mark': float(lc['hedges_mark']),
        'internal_mark_basis': mark_basis,
        'pure_execution': pure_exec,
        '  mid_to_touch': float(lc['mid_to_touch']),
        '  touch_to_exec': float(lc['touch_to_exec']),
        'unexplained': unexplained,
    }
    row = {
        'book': book,
        'volume_lots': vol,
        'abs_transfer_lots': float(lc['abs_transfer_lots']),
        'abs_hedge_lots': float(lc['abs_hedge_lots']),
        'library_m2m': float(lib['m2m_pnl']),
        'library_trade': float(lib['trade_pnl']),
        'overnight_check_diff': overnight - float(lc['overnight_check']),
        'pos_gap_pnl': float(lc['pos_gap_pnl']),
        'ttmds_fills_matched': int(lc['ttmds_fills_matched']),
        'ttmds_fills_total': int(lc['ttmds_fills_total']),
    }
    for k, v in dollar.items():
        row[f'{k}__$'] = v
        row[f'{k}__c_bbl'] = dollars_to_c_bbl(v, vol)
    combined.append(row)

detail = pd.DataFrame(combined).set_index('book').sort_index()

team = detail.sum(numeric_only=True)
team_vol = float(team['volume_lots'])
team_row = team.copy()
for comp in COMPONENT_ORDER:
    d = float(team[f'{comp}__$'])
    team_row[f'{comp}__c_bbl'] = dollars_to_c_bbl(d, team_vol)
detail.loc['TEAM TOTAL'] = team_row
detail[['volume_lots', 'total__$', 'overnight__$', 'market_risk__$', 'internal_mark_basis__$', 'pure_execution__$', 'unexplained__$']].round(2)


In [ ]:
def attribution_table(df: pd.DataFrame) -> pd.DataFrame:
    frames = []
    for book, row in df.iterrows():
        records = []
        for comp in COMPONENT_ORDER:
            records.append({
                'component': comp,
                '$': row[f'{comp}__$'],
                'c/bbl': row[f'{comp}__c_bbl'],
            })
        t = pd.DataFrame(records).set_index('component')
        t.columns = pd.MultiIndex.from_product([[book], t.columns])
        frames.append(t)
    return pd.concat(frames, axis=1)


summary = attribution_table(detail)
summary.round(2)


In [ ]:
# Sanity: library overnight vs lifecycle overnight; unexplained vs pos_gap
sanity = detail[[
    'overnight_check_diff', 'unexplained__$', 'pos_gap_pnl',
    'volume_lots', 'ttmds_fills_matched', 'ttmds_fills_total',
]].copy()
sanity['unexplained_vs_pos_gap'] = sanity['unexplained__$'] - sanity['pos_gap_pnl']
sanity.round(2)


In [ ]:
# Single-book focus
BOOK = 'Hedger Spreads'
if BOOK in detail.index:
    print('library: overnight / m2m / trade / gross')
    print(library.loc[BOOK, ['overnight', 'm2m_pnl', 'trade_pnl', 'library_gross']].round(2).to_string())
    print()
    summary[BOOK].round(2)
else:
    print(BOOK, 'not in detail — check library/lifecycle outputs above')
